# Trading 快速上手

本 notebook 演示 2.0 版本的两个新功能：
1. **动量轮动策略** —— `MomentumBasket` 和 `MomentumTopN`
2. **交互式 API** —— `run_backtest()` 直接返回结构化结果

运行本 notebook 需要：
- 已经通过 `fetcher.py` 把所有候选标的的历史行情同步到 MySQL
- `.env` 里配置好 `DB_*`

## 1. 最简调用：加权篮子

三个 ETF 各占 40%/40%/20%，动量低于 0 时该标的转现金。

In [ ]:
from trading import run_backtest, MomentumBasket, Schedule

result = run_backtest(
    strategy=MomentumBasket(
        weights={
            "159915.SZ": 0.4,
            "513100.SH": 0.4,
            "518880.SH": 0.2,
        },
        momentum_fn="simple_return",
        lookback=63,
        threshold=0.0,  # 动量为负则转现金
        schedule=Schedule.monthly_nth_trading_day(n=1),
    ),
    start="2022-01-01",
    initial_capital=50_000,
)

print(result.summary())

In [ ]:
# 交易明细
result.trades.head(20)

In [ ]:
# 当前下一次预期调仓动作
result.next_signal

In [ ]:
result.plot()

## 2. Top-N 等权

每月第 1 个交易日选动量最强的 1 个 ETF，预留 5% 现金 buffer。

In [ ]:
from trading import MomentumTopN

result = run_backtest(
    strategy=MomentumTopN(
        universe=["159915.SZ", "513100.SH", "518880.SH"],
        top_n=1,
        cash_buffer=0.05,
        momentum_fn="sharpe",     # 用风险调整后动量
        lookback=63,
        schedule=Schedule.monthly_nth_trading_day(n=1),
        cooldown_days=5,           # 5 个交易日内不连续调仓
    ),
    start="2022-01-01",
    initial_capital=50_000,
)
print(result.summary())

## 3. 动量算法对比

同样的 universe 和调仓表，比较不同动量定义。

In [ ]:
import pandas as pd

rows = []
for fn_name in ["simple_return", "log_return", "sharpe", "weighted", "dual_12_1"]:
    r = run_backtest(
        strategy=MomentumTopN(
            universe=["159915.SZ", "513100.SH", "518880.SH"],
            top_n=1,
            momentum_fn=fn_name,
            lookback=63 if fn_name != "dual_12_1" else 252,
            schedule=Schedule.monthly_nth_trading_day(n=1),
        ),
        start="2022-01-01",
        initial_capital=50_000,
    )
    rows.append(
        {
            "momentum_fn": fn_name,
            "年化收益率(%)": r.metrics["annual_return"],
            "最大回撤(%)": r.metrics["max_drawdown"],
            "夏普": r.metrics["sharpe_ratio"],
            "交易次数": r.metrics["total_trades"],
        }
    )
pd.DataFrame(rows).sort_values("夏普", ascending=False)